<a href="https://colab.research.google.com/github/vishav12/edumind-ai-tutor/blob/main/Authur_AI_Tutor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[link text](https://)# 🎓 EduMind AI Tutor
### Built by Vishav Bhan Singh Parmar
**An AI-powered tutor that understands PDFs, images, and text — and answers any study question.**

---
Run each cell one by one from top to bottom. At the end you will get a public URL — click it to open EduMind!

In [ ]:
# ============================================================
# CELL 1 — Install all required libraries
# ============================================================
!pip install -q streamlit pyngrok pymupdf Pillow google-generativeai langchain langchain-community faiss-cpu sentence-transformers pypdf
print('✅ All libraries installed successfully!')

In [ ]:

# ============================================================
# CELL 2 — Install Groq + Set API Key
# ============================================================
!pip install -q groq

import getpass, os
print('Enter your Groq API Key below (it will be hidden):')
GROQ_API_KEY = getpass.getpass('Groq API Key: ')
os.environ['GROQ_API_KEY'] = GROQ_API_KEY
print('✅ Groq API Key set successfully!')

In [ ]:
# ============================================================
# CELL 3 — Write EduMind app.py (Groq powered)
# ============================================================

cell3_lines = [
"import streamlit as st",
"from groq import Groq",
"import fitz",
"from PIL import Image",
"import os, base64",
"",
"st.set_page_config(page_title='Author AI Tutor', page_icon='🎓', layout='wide', initial_sidebar_state='expanded')",
"",
"st.markdown('''<style>",
"@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700&display=swap');",
"html,body,[class*='css']{font-family:'Inter',sans-serif;}",
"@keyframes fadeSlideIn{from{opacity:0;transform:translateY(20px)}to{opacity:1;transform:translateY(0)}}",
"@keyframes shimmer{0%{background-position:-200% center}100%{background-position:200% center}}",
"@keyframes float{0%,100%{transform:translateY(0)}50%{transform:translateY(-6px)}}",
"@keyframes pulse{0%,100%{box-shadow:0 0 0 0 rgba(99,102,241,.4)}50%{box-shadow:0 0 0 12px rgba(99,102,241,0)}}",
".hero-title{font-size:3rem;font-weight:700;text-align:center;background:linear-gradient(135deg,#6366f1,#8b5cf6,#06b6d4);background-size:200% auto;-webkit-background-clip:text;-webkit-text-fill-color:transparent;animation:fadeSlideIn .8s ease,shimmer 3s linear infinite;margin-bottom:.2rem;}",
".hero-sub{text-align:center;color:#94a3b8;font-size:1.1rem;animation:fadeSlideIn 1s ease;margin-bottom:2rem;}",
".feature-card{background:linear-gradient(135deg,#1e1b4b,#312e81);border:1px solid rgba(99,102,241,.3);border-radius:16px;padding:1.2rem 1rem;text-align:center;animation:fadeSlideIn 1.2s ease,float 4s ease-in-out infinite;color:white;margin-bottom:1rem;}",
".feature-card h3{font-size:1.5rem;margin:0;}",
".feature-card p{color:#c7d2fe;font-size:.9rem;margin:.3rem 0 0;}",
".chat-message-user{background:linear-gradient(135deg,#6366f1,#8b5cf6);color:white;padding:.9rem 1.2rem;border-radius:18px 18px 4px 18px;margin:.5rem 0;max-width:80%;margin-left:auto;animation:fadeSlideIn .4s ease;font-size:.95rem;}",
".chat-message-ai{background:rgba(30,27,75,.8);border:1px solid rgba(99,102,241,.25);color:#e2e8f0;padding:.9rem 1.2rem;border-radius:18px 18px 18px 4px;margin:.5rem 0;max-width:80%;animation:fadeSlideIn .4s ease;font-size:.95rem;line-height:1.6;}",
".upload-zone{border:2px dashed rgba(99,102,241,.5);border-radius:16px;padding:1.5rem;text-align:center;background:rgba(99,102,241,.05);margin-bottom:1rem;}",
".context-badge{background:rgba(6,182,212,.15);border:1px solid rgba(6,182,212,.4);border-radius:20px;padding:.3rem .9rem;font-size:.8rem;color:#06b6d4;display:inline-block;margin:.2rem;}",
".stButton>button{background:linear-gradient(135deg,#6366f1,#8b5cf6)!important;color:white!important;border:none!important;border-radius:12px!important;font-weight:600!important;animation:pulse 2s infinite;}",
"[data-testid='stAppViewContainer']{background:linear-gradient(160deg,#0f0c29,#1a1040,#0f0c29);}",
"[data-testid='stSidebar']{background:rgba(15,12,41,.95)!important;border-right:1px solid rgba(99,102,241,.2);}",
"label,.stMarkdown p,.stMarkdown li{color:#cbd5e1!important;}",
"</style>''', unsafe_allow_html=True)",
"",
"api_key = open('api_key.txt').read().strip() if os.path.exists('api_key.txt') else os.environ.get('GROQ_API_KEY', '')",
"if not api_key:",
"    st.error('Groq API key not found. Please run Cell 2 first.')",
"    st.stop()",
"client = Groq(api_key=api_key)",
"",
"if 'messages' not in st.session_state: st.session_state.messages = []",
"if 'context_text' not in st.session_state: st.session_state.context_text = ''",
"if 'context_names' not in st.session_state: st.session_state.context_names = []",
"if 'image_b64' not in st.session_state: st.session_state.image_b64 = []",
"",
"st.markdown('<div class=\"hero-title\">🎓 Author AI Tutor</div>', unsafe_allow_html=True)",
"st.markdown('<div class=\"hero-sub\">Upload anything. Ask anything. Learn everything.</div>', unsafe_allow_html=True)",
"",
"c1,c2,c3,c4 = st.columns(4)",
"with c1: st.markdown('<div class=\"feature-card\"><h3>📄</h3><p>PDF Upload</p></div>', unsafe_allow_html=True)",
"with c2: st.markdown('<div class=\"feature-card\"><h3>🖼️</h3><p>Image Analysis</p></div>', unsafe_allow_html=True)",
"with c3: st.markdown('<div class=\"feature-card\"><h3>💬</h3><p>Smart Chat</p></div>', unsafe_allow_html=True)",
"with c4: st.markdown('<div class=\"feature-card\"><h3>📚</h3><p>All Subjects</p></div>', unsafe_allow_html=True)",
"st.markdown('---')",
"",
"with st.sidebar:",
"    st.markdown('## 📂 Upload Your Study Material')",
"    st.markdown('<div class=\"upload-zone\">Drop your PDFs, images, or paste text below</div>', unsafe_allow_html=True)",
"    pdf_files = st.file_uploader('📄 Upload PDF(s)', type=['pdf'], accept_multiple_files=True)",
"    if pdf_files:",
"        for pdf in pdf_files:",
"            doc = fitz.open(stream=pdf.read(), filetype='pdf')",
"            text = ''.join(page.get_text() for page in doc)",
"            st.session_state.context_text += f'\\n\\n[PDF: {pdf.name}]\\n{text}'",
"            if pdf.name not in st.session_state.context_names: st.session_state.context_names.append(pdf.name)",
"        st.success(f'✅ {len(pdf_files)} PDF(s) loaded!')",
"    img_files = st.file_uploader('🖼️ Upload Image(s)', type=['png','jpg','jpeg','webp'], accept_multiple_files=True)",
"    if img_files:",
"        st.session_state.image_b64 = []",
"        for img in img_files:",
"            raw = img.read()",
"            b64 = base64.b64encode(raw).decode()",
"            st.session_state.image_b64.append(('image/jpeg', b64))",
"            if img.name not in st.session_state.context_names: st.session_state.context_names.append(img.name)",
"            st.image(raw, caption=img.name, use_column_width=True)",
"        st.success(f'✅ {len(img_files)} image(s) loaded!')",
"    st.markdown('### ✏️ Or paste your notes / text')",
"    manual_text = st.text_area('', placeholder='Paste your study notes here...', height=160)",
"    if st.button('Add Text to Context'):",
"        if manual_text.strip():",
"            st.session_state.context_text += f'\\n\\n[Manual Text]\\n{manual_text}'",
"            st.session_state.context_names.append('Manual text')",
"            st.success('✅ Text added!')",
"    if st.session_state.context_names:",
"        st.markdown('### 📌 Active Context')",
"        badges = ''.join(f'<span class=\"context-badge\">{n}</span>' for n in st.session_state.context_names)",
"        st.markdown(badges, unsafe_allow_html=True)",
"    if st.button('🗑️ Clear All Context'):",
"        st.session_state.context_text = ''",
"        st.session_state.context_names = []",
"        st.session_state.messages = []",
"        st.session_state.image_b64 = []",
"        st.rerun()",
"",
"st.markdown('## 💬 Ask Author Anything')",
"for msg in st.session_state.messages:",
"    if msg['role'] == 'user':",
"        st.markdown(f'<div class=\"chat-message-user\">🧑‍🎓 {msg[\"content\"]}</div>', unsafe_allow_html=True)",
"    else:",
"        st.markdown(f'<div class=\"chat-message-ai\">🎓 {msg[\"content\"]}</div>', unsafe_allow_html=True)",
"",
"col_input, col_btn = st.columns([5,1])",
"with col_input:",
"    user_question = st.text_input('', placeholder='Ask anything... e.g. Explain Newton laws, Summarize this PDF', label_visibility='collapsed')",
"with col_btn:",
"    send = st.button('Send ➤')",
"",
"if send and user_question.strip():",
"    st.session_state.messages.append({'role':'user','content':user_question})",
"    system_msg = {'role':'system','content':'You are Author, an expert AI tutor for ALL subjects — science, math, history, law, engineering, arts and more. Be clear, structured and encouraging. Use bullet points and examples where helpful. If study material is provided in the context, use it to answer accurately.'}",
"    context_section = ''",
"    if st.session_state.context_text.strip():",
"        context_section = f'\\n\\n--- UPLOADED STUDY MATERIAL ---\\n{st.session_state.context_text[:12000]}\\n--- END ---'",
"    user_content = []",
"    if st.session_state.image_b64:",
"        for mime, b64 in st.session_state.image_b64:",
"            user_content.append({'type':'image_url','image_url':{'url':f'data:{mime};base64,{b64}'}})",
"    user_content.append({'type':'text','text':f'{context_section}\\n\\nQuestion: {user_question}'})",
"    chat_history = [system_msg]",
"    for m in st.session_state.messages[-6:-1]:",
"        chat_history.append({'role':m['role'],'content':m['content']})",
"    chat_history.append({'role':'user','content':user_content})",
"    with st.spinner('🧠 Author is thinking...'):",
"        try:",
"            model_name = 'meta-llama/llama-4-scout-17b-16e-instruct' if not st.session_state.image_b64 else 'meta-llama/llama-4-scout-17b-16e-instruct'",
"            response = client.chat.completions.create(model=model_name, messages=chat_history, max_tokens=2048)",
"            answer = response.choices[0].message.content",
"        except Exception as e:",
"            answer = f'Sorry, error: {str(e)}'",
"    st.session_state.messages.append({'role':'assistant','content':answer})",
"    st.rerun()",
"",
"st.markdown('---')",
"st.markdown('<p style=\"text-align:center;color:#475569;font-size:.8rem;\">Author AI Tutor • Built by Vishav Bhan Singh Parmar • Powered by Groq + Llama 4</p>', unsafe_allow_html=True)",
]

with open('app.py', 'w') as f:
    f.write('\n'.join(cell3_lines))

print('✅ app.py written successfully with Groq + Llama 4!')

In [ ]:
# Install cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

import subprocess, threading, time, re

def run_cloudflared():
    global tunnel_url
    process = subprocess.Popen(
        ['./cloudflared', 'tunnel', '--url', 'http://localhost:8501'],
        stderr=subprocess.PIPE, stdout=subprocess.PIPE
    )
    for line in process.stderr:
        line = line.decode()
        match = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', line)
        if match:
            tunnel_url = match.group(0)
            print(f"\n{'='*55}")
            print("🌐 EduMind is available at:")
            print(f"   {tunnel_url}")
            print(f"{'='*55}")
            break

tunnel_url = ""
t = threading.Thread(target=run_cloudflared, daemon=True)
t.start()
time.sleep(5)
print("👆 Use the URL above after running Cell 5!")


In [ ]:
# ============================================================
# CELL 5 — Launch EduMind!
# ============================================================
print('🚀 Launching EduMind AI Tutor...')
print('The app is starting — use the ngrok URL from Cell 4!')
!streamlit run app.py --server.port 8501 --server.headless true

In [ ]:
# ============================================================
# CELL 3b — Save API key to a file so Streamlit can read it
# ============================================================
import os

groq_key = os.environ.get('GROQ_API_KEY', '')
with open('api_key.txt', 'w') as f:
    f.write(groq_key)

print('✅ API key saved successfully!')

✅ API key saved successfully!
